In [1]:
import os
os.environ['XLA_PYTHON_CLIENT_PREALLOCATE'] = 'false'
from collections import namedtuple
import chex
from src.maenv.util import notify
import jax
import jax.numpy as jnp
from src.maenv.physics import Transform, RigidBody, CircleCollider


class UnitStatus(namedtuple('UnitStatus', ['id', 'health', 'attack_damage', 'attack_range', 'attack_cooldown', 'cooldown', 'sight_angle', 'sight_radius'])):
    id: chex.Array
    health: chex.Array
    attack_damage: chex.Array
    attack_range: chex.Array
    attack_cooldown: chex.Array
    cooldown: chex.Array
    sight_angle: chex.Array
    sight_radius: chex.Array

move_table = jnp.array(
        [
            [0, 1.0],
            [0, -1.0],
            [-1.0, 0],
            [1.0, 0],
            [0.0, 0.0],
            [0.0, 0.0],
        ]
    )


class UnitAction:
    UP = 0
    DOWN = 1
    LEFT = 2
    RIGHT = 3
    ATTACK = 4
    IDLE = 5

class DefaultUnit(namedtuple('DefaultUnit', ['transform', 'rigidbody', 'collider', 'team', 'pos_limit', 'status'])):
    transform: Transform
    rigidbody: RigidBody
    collider: CircleCollider
    team: chex.Array
    pos_limit: chex.Array
    status: UnitStatus
    
    def __new__(cls, transform, rigidbody, collider, team, pos_limit, status):
        return super().__new__(cls, transform, rigidbody, collider, team, pos_limit, status)
    
    def act(self, objects, action):
        # action : [rotate_angle, discrete action]
        is_attack = UnitAction.ATTACK == action[1]
        return notify(objects, 'hit', (action, self, is_attack))
    
    def on_hit(self, objects, info):
        attacker: DefaultUnit
        action, attacker, is_attack = info
        in_range = jnp.abs(attacker.transform.position - self.transform.position).sum() < attacker.status.attack_range #TODO : consider the angle
        is_enemy = attacker.team != self.team
        is_target = in_range & is_enemy & is_attack
        damaged_status = self.on_damage(attacker.status.attack_damage)
        new_status = jax.tree.map(lambda x, y: jnp.where(is_target, y, x), self.status, damaged_status)
        return self._replace(status=new_status)
        
    def on_damage(self, damage) -> UnitStatus:
        return self.status._replace(health=self.status.health - damage)
    
        

In [2]:
unit1 = DefaultUnit(
    transform=Transform(position=jnp.array([0.0, 0.0]), rotation=jnp.array([0.0])),
    rigidbody=RigidBody(mass=jnp.array([1.0]), velocity=jnp.array([0.0, 0.0]), acceleration=jnp.array([0.0, 0.0]), is_kinematic=jnp.array([False])),
    collider=CircleCollider(radius=1.0),
    team=jnp.array([0]),
    pos_limit=jnp.array([-10.0, 10.0]),
    status=UnitStatus(id=jnp.array([0]), health=jnp.array([100.0]), attack_damage=jnp.array([10.0]), attack_range=jnp.array([1.0]), attack_cooldown=jnp.array([0.0]), cooldown=jnp.array([0.0]), sight_angle=jnp.array([0.0]), sight_radius=jnp.array([10.0]))
)

unit2 = DefaultUnit(
    transform=Transform(position=jnp.array([0.0, 0.0]), rotation=jnp.array([0.0])),
    rigidbody=RigidBody(mass=jnp.array([1.0]), velocity=jnp.array([0.0, 0.0]), acceleration=jnp.array([0.0, 0.0]), is_kinematic=jnp.array([False])),
    collider=CircleCollider(radius=1.0),
    team=jnp.array([1]),
    pos_limit=jnp.array([-10.0, 10.0]),
    status=UnitStatus(id=jnp.array([1]), health=jnp.array([100.0]), attack_damage=jnp.array([10.0]), attack_range=jnp.array([1.0]), attack_cooldown=jnp.array([0.0]), cooldown=jnp.array([0.0]), sight_angle=jnp.array([0.0]), sight_radius=jnp.array([10.0]))
)

In [3]:
objecs = {
    'unit1' : unit1,
    'unit2' : unit2,
}

In [4]:
from src.maenv.base_maenv import BaseMAEnv
from src.maenv.tabs.tabs_battle_simulator.battle_simulator import DefaultUnit, UnitStatus, GameManager
from src.maenv.physics import Transform, RigidBody, CircleCollider, physics_step
from easydict import EasyDict
from typing import Dict
import jax.numpy as jnp

class TABS(BaseMAEnv):
    def __init__(self, num_agents: int = 4, physics_config: Dict[str, float] = EasyDict(
            {"dt": 0.2, "percent": 0.5, "slop": 0.01, "restitution": 0.8}
        ),
    ):
        super().__init__(num_agents, physics_config)

    def get_obs(self, state):
        return jnp.zeros((1, 1))
    
    
    def reset(self, key):
        unit1 = DefaultUnit(
        transform=Transform(position=jnp.array([0.0, 0.0]), rotation=jnp.array([0.0])),
        rigidbody=RigidBody(
            mass=jnp.array([1.0]),
            velocity=jnp.array([0.0, 0.0]),
            acceleration=jnp.array([0.0, 0.0]),
            is_kinematic=jnp.array([False]),
        ),
            collider=CircleCollider(radius=1.0),
            team=jnp.array([0]),
            pos_limit=jnp.array([-10.0, 10.0]),
            status=UnitStatus(
                id=jnp.array([0]),
                health=jnp.array([100.0]),
                attack_damage=jnp.array([10.0]),
                attack_range=jnp.array([3.0]),
                attack_cooldown=jnp.array([2.0]),
                cooldown=jnp.array([0.0]),
                sight_angle=jnp.array([0.0]),
                sight_radius=jnp.array([5.0]),
            ),
            attacking=jnp.array([False]),
        )

        unit2 = DefaultUnit(
            transform=Transform(position=jnp.array([5.0, 3.0]), rotation=jnp.array([2.3])),
            rigidbody=RigidBody(
                mass=jnp.array([1.0]),
                velocity=jnp.array([0.0, 0.0]),
                acceleration=jnp.array([0.0, 0.0]),
                is_kinematic=jnp.array([False]),
            ),
            collider=CircleCollider(radius=1.5),
            team=jnp.array([1]),
            pos_limit=jnp.array([-10.0, 10.0]),
            status=UnitStatus(
                id=jnp.array([1]),
                health=jnp.array([80.0]),
                attack_damage=jnp.array([15.0]),
                attack_range=jnp.array([10.0]),
                attack_cooldown=jnp.array([1.5]),
                cooldown=jnp.array([0.0]),
                sight_angle=jnp.array([0.0]),
                sight_radius=jnp.array([6.0]),
            ),
            attacking=jnp.array([True]),
        )
        unit3 = DefaultUnit(
            transform=Transform(position=jnp.array([-5.0, -3.0]), rotation=jnp.array([2.3])),
            rigidbody=RigidBody(
                mass=jnp.array([1.0]),
                velocity=jnp.array([0.0, 0.0]),
                acceleration=jnp.array([0.0, 0.0]),
                is_kinematic=jnp.array([False]),
            ),
            collider=CircleCollider(radius=1.5),
            team=jnp.array([1]),
            pos_limit=jnp.array([-10.0, 10.0]),
            status=UnitStatus(
                id=jnp.array([1]),
                health=jnp.array([80.0]),
                attack_damage=jnp.array([15.0]),
                attack_range=jnp.array([10.0]),
                attack_cooldown=jnp.array([1.5]),
                cooldown=jnp.array([0.0]),
                sight_angle=jnp.array([0.0]),
                sight_radius=jnp.array([6.0]),
            ),
            attacking=jnp.array([True]),
        )
        
        game_manager = GameManager(
            reward=jnp.array([0.0]),
            done=jnp.array([False]),
            timestep=jnp.array([0]),
            target=jnp.array([-1]),
        )
        
        state = {
            "unit1": unit1,
            "unit2": unit2,
            "unit3": unit3,
            "game_manager": game_manager,
        }
        
        return self.get_obs(state), state
    
    
    
    
    def step(self, key, state, action):
        for sprite in state.keys():
            if hasattr(state[sprite], "update"):
                state[sprite] = state[sprite].update(config=self.physics_config)

        collider_filter = {
            "unit1": ["unit2"],
            "unit2": ["unit1"],
        }
        
        state = physics_step(self.physics_config, state, list(state.keys()), collider_filter)
        for sprite in ["unit1", "unit2", "unit3"]:
            state[sprite] = state[sprite].act(state, action[sprite])
        
        
        
        return self.get_obs(state), state, 0.0, False, {'timestep': 0}
    
    def render(self, state):
        return None
    


In [6]:
import pygame
import sys
import jax.numpy as jnp
import numpy as np
from typing import Dict, Any
import math
from src.maenv.tabs.tabs_battle_simulator.battle_simulator import GameManager
from src.maenv.physics import physics_step

class PygameRenderer:
    """실시간으로 게임 객체들을 렌더링하는 pygame 렌더러"""

    def __init__(self, width=800, height=600, fps=60, world_scale=20):
        """
        초기화

        Args:
            width: 창 너비
            height: 창 높이
            fps: 프레임 레이트
            world_scale: 게임 월드 좌표를 픽셀로 변환하는 스케일
        """
        pygame.init()

        self.width = width
        self.height = height
        self.fps = fps
        self.world_scale = world_scale

        # 화면 설정
        self.screen = pygame.display.set_mode((width, height))
        pygame.display.set_caption("TABS Battle Renderer")

        # 클럭 설정
        self.clock = pygame.time.Clock()

        # 색상 정의
        self.colors = {
            "background": (50, 50, 50),
            "team_0": (255, 100, 100),  # 빨간색 팀
            "team_1": (100, 100, 255),  # 파란색 팀
            "health_bg": (200, 200, 200),  # 체력바 배경
            "health_fg": (100, 255, 100),  # 체력바 전경
            "attack_range": (255, 255, 0, 50),  # 공격 범위 (반투명 노란색)
            "sight_range": (0, 255, 255, 30),  # 시야 범위 (반투명 청록색)
            "attacking": (255, 255, 255, 100),  # 공격 중 표시
        }

        # 폰트 설정
        self.font = pygame.font.Font(None, 24)
        self.small_font = pygame.font.Font(None, 16)

        # 카메라 설정
        self.camera_x = 0
        self.camera_y = 0
        self.zoom = 1.0

        self.running = True

    def world_to_screen(self, world_pos):
        """월드 좌표를 스크린 좌표로 변환"""
        world_x, world_y = world_pos
        screen_x = (world_x - self.camera_x) * self.world_scale * self.zoom + self.width // 2
        screen_y = (world_y - self.camera_y) * self.world_scale * self.zoom + self.height // 2
        return int(screen_x), int(screen_y)

    def handle_events(self):
        """이벤트 처리"""
        for event in pygame.event.get():
            if event.type == pygame.QUIT:
                self.running = False
            elif event.type == pygame.KEYDOWN:
                if event.key == pygame.K_ESCAPE:
                    self.running = False
                elif event.key == pygame.K_SPACE:
                    # 스페이스바로 카메라 리셋
                    self.camera_x = 0
                    self.camera_y = 0
                    self.zoom = 1.0
            elif event.type == pygame.MOUSEWHEEL:
                # 마우스 휠로 줌
                zoom_factor = 1.1
                if event.y > 0:
                    self.zoom *= zoom_factor
                else:
                    self.zoom /= zoom_factor
                self.zoom = max(0.1, min(5.0, self.zoom))

        # 키보드로 카메라 이동
        keys = pygame.key.get_pressed()
        move_speed = 0.5 / self.zoom
        if keys[pygame.K_w] or keys[pygame.K_UP]:
            self.camera_y -= move_speed
        if keys[pygame.K_s] or keys[pygame.K_DOWN]:
            self.camera_y += move_speed
        if keys[pygame.K_a] or keys[pygame.K_LEFT]:
            self.camera_x -= move_speed
        if keys[pygame.K_d] or keys[pygame.K_RIGHT]:
            self.camera_x += move_speed

    def draw_unit(self, unit_name, unit, show_ranges=True):
        """개별 유닛을 그리기"""
        try:
            # JAX 배열에서 numpy 배열로 변환
            if hasattr(unit.transform.position, "__array__"):
                pos = np.array(unit.transform.position)
            else:
                pos = unit.transform.position

            if hasattr(unit.transform.rotation, "__array__"):
                rotation = float(np.array(unit.transform.rotation))
            else:
                rotation = float(unit.transform.rotation)

            if hasattr(unit.collider.radius, "__array__"):
                radius = float(np.array(unit.collider.radius))
            else:
                radius = float(unit.collider.radius)

            if hasattr(unit.team, "__array__"):
                team = int(np.array(unit.team))
            else:
                team = int(unit.team)

            if hasattr(unit.status.health, "__array__"):
                health = float(np.array(unit.status.health))
            else:
                health = float(unit.status.health)

            if hasattr(unit.status.attack_range, "__array__"):
                attack_range = float(np.array(unit.status.attack_range))
            else:
                attack_range = float(unit.status.attack_range)

            if hasattr(unit.status.sight_radius, "__array__"):
                sight_radius = float(np.array(unit.status.sight_radius))
            else:
                sight_radius = float(unit.status.sight_radius)

            # 공격 중인지 확인
            attacking = False
            if hasattr(unit, "attacking"):
                if hasattr(unit.attacking, "__array__"):
                    attacking = bool(np.array(unit.attacking))
                else:
                    attacking = bool(unit.attacking)

            screen_pos = self.world_to_screen(pos)
            screen_radius = int(radius * self.world_scale * self.zoom)

            # 화면에 보이는지 확인
            if (
                screen_pos[0] < -screen_radius
                or screen_pos[0] > self.width + screen_radius
                or screen_pos[1] < -screen_radius
                or screen_pos[1] > self.height + screen_radius
            ):
                return

            # 범위 표시 (옵션)
            if show_ranges and self.zoom > 0.3:
                # 시야 범위 (원형)
                sight_screen_radius = int(sight_radius * self.world_scale * self.zoom)
                if sight_screen_radius > 5:
                    sight_surface = pygame.Surface(
                        (sight_screen_radius * 2, sight_screen_radius * 2), pygame.SRCALPHA
                    )
                    pygame.draw.circle(
                        sight_surface,
                        self.colors["sight_range"],
                        (sight_screen_radius, sight_screen_radius),
                        sight_screen_radius,
                    )
                    self.screen.blit(
                        sight_surface,
                        (screen_pos[0] - sight_screen_radius, screen_pos[1] - sight_screen_radius),
                    )

                # 공격 범위 (직사각형) - TABS 스타일
                self.draw_rectangular_attack_range(screen_pos, rotation, attack_range, radius)

            # 유닛 몸체
            team_color = self.colors.get(f"team_{team}", (128, 128, 128))
            pygame.draw.circle(self.screen, team_color, screen_pos, max(1, screen_radius))

            # 방향 표시 (회전 정보가 있을 때)
            if screen_radius > 3:
                direction_end = (
                    screen_pos[0] + int(math.cos(rotation) * screen_radius * 0.8),
                    screen_pos[1] + int(math.sin(rotation) * screen_radius * 0.8),
                )
                pygame.draw.line(self.screen, (255, 255, 255), screen_pos, direction_end, 2)

            # 공격 중 표시
            if attacking and screen_radius > 2:
                attack_surface = pygame.Surface(
                    (screen_radius * 3, screen_radius * 3), pygame.SRCALPHA
                )
                pygame.draw.circle(
                    attack_surface,
                    self.colors["attacking"],
                    (screen_radius * 1.5, screen_radius * 1.5),
                    screen_radius * 1.5,
                )
                self.screen.blit(
                    attack_surface,
                    (screen_pos[0] - screen_radius * 1.5, screen_pos[1] - screen_radius * 1.5),
                )

            # 체력바 (줌이 충분할 때만)
            if self.zoom > 0.5 and screen_radius > 5:
                health_bar_width = screen_radius * 2
                health_bar_height = 4
                health_bar_y = screen_pos[1] - screen_radius - health_bar_height - 2

                # 최대 체력 추정 (일반적으로 100)
                max_health = 100.0
                if hasattr(unit.status, "max_health"):
                    max_health = float(np.array(unit.status.max_health))

                health_ratio = max(0, min(1, health / max_health))

                # 체력바 배경
                pygame.draw.rect(
                    self.screen,
                    self.colors["health_bg"],
                    (
                        screen_pos[0] - health_bar_width // 2,
                        health_bar_y,
                        health_bar_width,
                        health_bar_height,
                    ),
                )

                # 체력바 전경
                if health_ratio > 0:
                    pygame.draw.rect(
                        self.screen,
                        self.colors["health_fg"],
                        (
                            screen_pos[0] - health_bar_width // 2,
                            health_bar_y,
                            int(health_bar_width * health_ratio),
                            health_bar_height,
                        ),
                    )

            # 유닛 이름 및 정보 (줌이 충분할 때만)
            if self.zoom > 1.0 and screen_radius > 10:
                # 유닛 이름
                name_text = self.small_font.render(unit_name, True, (255, 255, 255))
                name_rect = name_text.get_rect(
                    center=(screen_pos[0], screen_pos[1] + screen_radius + 15)
                )
                self.screen.blit(name_text, name_rect)

                # 체력 수치
                health_text = self.small_font.render(f"{int(health)}", True, (255, 255, 255))
                health_rect = health_text.get_rect(
                    center=(screen_pos[0], screen_pos[1] + screen_radius + 30)
                )
                self.screen.blit(health_text, health_rect)

        except Exception as e:
            print(f"Error drawing unit {unit_name}: {e}")

    def draw_ui(self, objects):
        """UI 정보 그리기"""
        y_offset = 10

        # 컨트롤 정보
        controls = [
            "Controls:",
            "WASD/Arrow Keys: Move Camera",
            "Mouse Wheel: Zoom",
            "Space: Reset Camera",
            "ESC: Exit",
        ]

        for i, text in enumerate(controls):
            color = (255, 255, 255) if i == 0 else (200, 200, 200)
            rendered_text = self.small_font.render(text, True, color)
            self.screen.blit(rendered_text, (10, y_offset))
            y_offset += 20

        # 게임 정보
        y_offset += 10
        unit_count = sum(1 for key in objects.keys() if "unit" in key.lower())
        info_text = self.font.render(f"Units: {unit_count}", True, (255, 255, 255))
        self.screen.blit(info_text, (10, y_offset))

        y_offset += 25
        zoom_text = self.small_font.render(f"Zoom: {self.zoom:.2f}", True, (255, 255, 255))
        self.screen.blit(zoom_text, (10, y_offset))

        # 범례
        legend_x = self.width - 150
        legend_y = 10

        legend_items = [
            ("Team 0", self.colors["team_0"]),
            ("Team 1", self.colors["team_1"]),
            ("Attack Range", self.colors["attack_range"][:3]),
            ("Sight Range", self.colors["sight_range"][:3]),
        ]

        legend_title = self.small_font.render("Legend:", True, (255, 255, 255))
        self.screen.blit(legend_title, (legend_x, legend_y))
        legend_y += 20

        for name, color in legend_items:
            pygame.draw.circle(self.screen, color, (legend_x + 10, legend_y + 8), 6)
            text = self.small_font.render(name, True, (255, 255, 255))
            self.screen.blit(text, (legend_x + 25, legend_y))
            legend_y += 18

    def render(self, objects: Dict[str, Any], show_ranges=True):
        """
        객체들을 렌더링

        Args:
            objects: 게임 객체들의 딕셔너리 (key는 이름, value는 unit 객체)
            show_ranges: 공격/시야 범위를 표시할지 여부
        """
        if not self.running:
            return False

        # 이벤트 처리
        self.handle_events()

        # 화면 클리어
        self.screen.fill(self.colors["background"])

        # 격자 그리기 (옵션)
        if self.zoom > 0.5:
            self.draw_grid()

        # 유닛들 그리기
        for obj_name, obj in objects.items():
            if "unit" in obj_name.lower() and hasattr(obj, "transform"):
                self.draw_unit(obj_name, obj, show_ranges)

        # UI 그리기
        self.draw_ui(objects)

        # 화면 업데이트
        pygame.display.flip()
        self.clock.tick(self.fps)

        return self.running

    def draw_rectangular_attack_range(
        self, screen_pos, rotation, attack_range, unit_radius, attack_range_angle=math.pi / 4
    ):
        """
        TABS 스타일의 직사각형 공격 범위를 그리기

        Args:
            screen_pos: 유닛의 화면 좌표
            rotation: 유닛의 회전 각도 (라디안)
            attack_range: 공격 범위 거리
            unit_radius: 유닛의 반지름
            attack_range_angle: 공격 각도의 절반 (기본값: π/4)
        """
        if attack_range * self.world_scale * self.zoom < 10:
            return

        # 공격 범위의 스크린 크기 계산
        attack_screen_range = attack_range * self.world_scale * self.zoom
        unit_screen_radius = unit_radius * self.world_scale * self.zoom

        # 공격 범위 직사각형의 크기 계산
        cos_half_angle = math.cos(attack_range_angle / 2)
        sin_half_angle = math.sin(attack_range_angle / 2)

        # 직사각형의 높이는 공격 각도에 따라 결정
        rect_height = 2 * unit_screen_radius * sin_half_angle / cos_half_angle
        rect_width = attack_screen_range

        # 직사각형의 로컬 좌표 (유닛 중심에서 앞으로 뻗어나가는 형태)
        rect_points = [
            (0, -rect_height / 2),  # 좌측 후방
            (rect_width, -rect_height / 2),  # 좌측 전방
            (rect_width, rect_height / 2),  # 우측 전방
            (0, rect_height / 2),  # 우측 후방
        ]

        # 회전 변환 적용
        cos_rot = math.cos(rotation)
        sin_rot = math.sin(rotation)

        rotated_points = []
        for x, y in rect_points:
            # 회전 행렬 적용
            new_x = x * cos_rot - y * sin_rot
            new_y = x * sin_rot + y * cos_rot
            # 유닛 위치에 상대적으로 배치
            screen_x = screen_pos[0] + new_x
            screen_y = screen_pos[1] + new_y
            rotated_points.append((screen_x, screen_y))

        # 반투명 직사각형 그리기
        try:
            # 충분한 크기의 surface 생성
            max_dim = max(self.width, self.height)
            attack_surface = pygame.Surface((max_dim, max_dim), pygame.SRCALPHA)

            # 상대 좌표로 변환 (surface 중심 기준)
            surface_center = max_dim // 2
            surface_points = []
            for x, y in rotated_points:
                surface_x = x - screen_pos[0] + surface_center
                surface_y = y - screen_pos[1] + surface_center
                surface_points.append((surface_x, surface_y))

            # 다각형 그리기
            pygame.draw.polygon(attack_surface, self.colors["attack_range"], surface_points)

            # 외곽선 그리기
            pygame.draw.polygon(
                attack_surface, (*self.colors["attack_range"][:3], 150), surface_points, 2
            )

            # 화면에 블릿
            blit_x = screen_pos[0] - surface_center
            blit_y = screen_pos[1] - surface_center
            self.screen.blit(attack_surface, (blit_x, blit_y))

        except Exception as e:
            # 에러 발생시 간단한 원형으로 대체
            attack_screen_radius = int(attack_range * self.world_scale * self.zoom)
            if attack_screen_radius > 3:
                attack_surface = pygame.Surface(
                    (attack_screen_radius * 2, attack_screen_radius * 2), pygame.SRCALPHA
                )
                pygame.draw.circle(
                    attack_surface,
                    self.colors["attack_range"],
                    (attack_screen_radius, attack_screen_radius),
                    attack_screen_radius,
                )
                self.screen.blit(
                    attack_surface,
                    (screen_pos[0] - attack_screen_radius, screen_pos[1] - attack_screen_radius),
                )

    def draw_grid(self):
        """격자 그리기"""
        grid_spacing = 5  # 월드 단위
        grid_color = (70, 70, 70)

        # 화면에 보이는 격자 범위 계산
        left = self.camera_x - (self.width // 2) / (self.world_scale * self.zoom)
        right = self.camera_x + (self.width // 2) / (self.world_scale * self.zoom)
        top = self.camera_y - (self.height // 2) / (self.world_scale * self.zoom)
        bottom = self.camera_y + (self.height // 2) / (self.world_scale * self.zoom)

        # 세로선
        start_x = int(left // grid_spacing) * grid_spacing
        x = start_x
        while x <= right:
            screen_x, _ = self.world_to_screen((x, 0))
            if 0 <= screen_x <= self.width:
                pygame.draw.line(self.screen, grid_color, (screen_x, 0), (screen_x, self.height))
            x += grid_spacing

        # 가로선
        start_y = int(top // grid_spacing) * grid_spacing
        y = start_y
        while y <= bottom:
            _, screen_y = self.world_to_screen((0, y))
            if 0 <= screen_y <= self.height:
                pygame.draw.line(self.screen, grid_color, (0, screen_y), (self.width, screen_y))
            y += grid_spacing

    def close(self):
        """렌더러 종료"""
        pygame.quit()


# 사용 예시 함수
def render_loop(state, fps=60, show_ranges=True):
    """
    실시간 렌더링 루프

    Args:
        objects: 게임 객체들의 딕셔너리
        fps: 프레임 레이트
        show_ranges: 공격/시야 범위 표시 여부
    """
    renderer = PygameRenderer(fps=fps)

    try:
        while renderer.render(state, show_ranges):
            # 여기서 게임 로직 업데이트를 할 수 있습니다
            # objects = update_game_logic(objects)
            obs, state, reward, done, info = env.step(jax.random.key(0), state, {"unit1": jnp.array([np.random.uniform(-0.1, 0.1), np.random.randint(0, 4)]), "unit2": jnp.array([np.random.uniform(-0.1, 0.1), np.random.randint(0, 4)]), "unit3": jnp.array([np.random.uniform(-0.1, 0.1), np.random.randint(0, 4)])})

            pass
    finally:
        renderer.close()


if __name__ == "__main__":
    # 테스트용 더미 데이터
    import jax.numpy as jnp
    from src.maenv.physics import Transform, RigidBody, CircleCollider
    from src.maenv.tabs.tabs_battle_simulator.battle_simulator import DefaultUnit, UnitStatus

    # 테스트 유닛 생성
    env = TABS()
    obs, state = env.reset(jax.random.key(0))

    print("Starting test renderer...")
    print("Controls: WASD to move camera, mouse wheel to zoom, space to reset, ESC to exit")
    render_loop(state)


Starting test renderer...
Controls: WASD to move camera, mouse wheel to zoom, space to reset, ESC to exit


C:\Users\user\AppData\Local\Temp\ipykernel_23812\3374241783.py:111: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  rotation = float(np.array(unit.transform.rotation))
C:\Users\user\AppData\Local\Temp\ipykernel_23812\3374241783.py:121: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  team = int(np.array(unit.team))
C:\Users\user\AppData\Local\Temp\ipykernel_23812\3374241783.py:126: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  health = float(np.array(unit.status.health)